# 03 · 数据质量与边界测试

先运行01、02。所有测试断言留在本 notebook，计算逻辑在 src/project。

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if (ROOT / 'project' / 'pyproject.toml').exists(): ROOT = ROOT / 'project'
assert (ROOT / 'configs' / 'commodity_map.csv').exists(), 'Open the project folder first'
sys.path.insert(0, str(ROOT / 'src'))
from project.paths import ensure_directories
ensure_directories()
pd.set_option('display.max_columns', 14)


In [ ]:
from project.data import load_reference,construct_monthly,momentum_signal
from project.notice_conventions import notice_convention
panel=pd.read_parquet(ROOT/'data/processed/commodity_monthly.parquet')
reference=load_reference()
assert len(reference)==12207 and reference.paper_code.nunique()==29
assert reference.slope.isna().sum()==2
assert not panel.duplicated(['paper_code','date']).any()
chosen=panel[panel.selected_futcode.notna()]
assert chosen.notice_cutoff.gt(chosen.date).all()
valid=chosen.dropna(subset=['excess_return'])
np.testing.assert_allclose(valid.excess_return,valid.exit_price/valid.entry_price-1)
assert valid.entry_quote_date.le(valid.formation_date).all()
assert valid.exit_quote_date.le(valid.date).all()
assert valid.mapping_status.eq('verified').all()
assert panel.notice_policy.eq('appendix_conventions').all()
assert notice_convention('CL',pd.Timestamp('2011-05-01'))[0]==pd.Timestamp('2011-04-25')
assert notice_convention('CT',pd.Timestamp('2011-05-01'))[0]==pd.Timestamp('2011-05-15')
assert notice_convention('FC',pd.Timestamp('2011-05-01'))[0]==pd.Timestamp('2011-05-26')
print('Keys, prices, timing and Table-I convention checks passed')


## 有区分力的合成测试

三份合约在 t 月末报价，第二份可安全持有：应选第二份而不是“安全集合的第二份”；更换下一月其他合约的价格不能改变选约。缺少所选合约的退出价不能偷偷切换另一份。日历中断不得压缩六个月动量。


In [ ]:
def synthetic_snapshot():
    rows=[]
    for date, prices in [(pd.Timestamp('2020-01-31'),[90.,100.,120.]),
                         (pd.Timestamp('2020-02-29'),[999.,110.,500.])]:
        for i, price in enumerate(prices,1):
            rows.append(dict(contrcode=1,date=date,futcode=i,paper_code='X',commodity='X',
                sector='energy',priority=1,status='verified',
                maturity=pd.Timestamp(2020,2+i,1),expiry=pd.Timestamp(2020,2+i,15),
                notice_cutoff=pd.Timestamp(2020,2+i,1),firstnoticedate=pd.Timestamp(2020,2+i,1),
                notice_source='vendor_fnd',quote_age_days=0,quote_date=date,
                settlement=price,open_interest=10.))
    return pd.DataFrame(rows)
s=synthetic_snapshot()
test=construct_monthly(s).iloc[0]
assert test.selected_futcode==2
assert np.isclose(test.excess_return,.1)
changed=s.copy(); changed.loc[(changed.date.eq('2020-02-29')) & changed.futcode.ne(2),'settlement']=1e6
assert construct_monthly(changed).iloc[0].selected_futcode==2
missing=s[~(s.date.eq('2020-02-29') & s.futcode.eq(2))]
assert pd.isna(construct_monthly(missing).iloc[0].excess_return)
dates=pd.date_range('2020-01-31',periods=9,freq='ME').delete(3)
gapped=pd.DataFrame({'date':dates,'paper_code':'X','excess_return':.01})
assert momentum_signal(gapped).empty
print('Contract selection, no future-price selection, missing exit and calendar-gap tests: passed')


## 公布日对齐与持仓缺失测试

构造一次延迟公布：2 月的报告 4 月才公布，因此不能在 2 月或 3 月使用。若没有官方日历，实际 CFTC 预测版数据保持未生成。


In [ ]:
from project.positioning import align_releases
weekly=pd.DataFrame({'paper_code':['X','X'],
    'report_date':pd.to_datetime(['2020-01-21','2020-02-25']),
    'net_noncommercial_share':[.1,.9]})
releases=pd.DataFrame({'report_date':weekly.report_date,
                      'release_date':pd.to_datetime(['2020-01-24','2020-04-03'])})
aligned=align_releases(weekly,releases)
assert aligned.loc[aligned.date.eq('2020-03-31'),'net_noncommercial_share'].iloc[0]==.1
assert aligned.loc[aligned.date.eq('2020-04-30'),'net_noncommercial_share'].iloc[0]==.9
print('Publication-delay test: passed')
audit=pd.read_csv(ROOT / 'data/interim/wrds_quote_audit.csv')
display(audit.sum(numeric_only=True).drop('contrcode'))
